In [ ]:
"""
Healthcare ML Pipeline — Diabetes Prediction
=============================================
Models: Logistic Regression | Decision Tree | Random Forest
Dataset: Synthetic patient data (n=1000, ~35% diabetic)
Target: Diabetes diagnosis (0 = No, 1 = Yes)

Requirements:
    pip install numpy pandas scikit-learn matplotlib seaborn
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve,
    confusion_matrix, classification_report
)

# ─────────────────────────────────────────────
# 1. GENERATE SYNTHETIC HEALTHCARE DATASET
# ─────────────────────────────────────────────

np.random.seed(42)
N_POS = 350   # diabetic patients
N_NEG = 650   # non-diabetic patients

def sample_group(n, diabetic: bool):
    """Sample clinical features with class-conditional distributions."""
    d = int(diabetic)
    age              = np.random.normal(55 if d else 44, 12, n).clip(18, 85)
    bmi              = np.random.normal(32 if d else 26, 5,  n).clip(15, 55)
    glucose          = np.random.normal(155 if d else 95, 25, n).clip(60, 300)
    blood_pressure   = np.random.normal(85 if d else 75, 12, n).clip(50, 130)
    hba1c            = np.random.normal(7.5 if d else 5.2, 1.0, n).clip(4.0, 14.0)
    insulin_res      = np.random.normal(6.5 if d else 3.5, 1.5, n).clip(0, 20)
    family_history   = np.random.binomial(1, 0.55 if d else 0.20, n)
    physical_activity= np.random.normal(2.0 if d else 4.5, 1.5, n).clip(0, 7)
    diabetes         = np.full(n, d, dtype=int)
    return pd.DataFrame({
        "age": age.round(1),
        "bmi": bmi.round(1),
        "glucose_level": glucose.round(1),
        "blood_pressure": blood_pressure.round(1),
        "hba1c": hba1c.round(2),
        "insulin_resistance": insulin_res.round(2),
        "family_history": family_history,
        "physical_activity": physical_activity.round(1),
        "diabetes": diabetes,
    })

df = pd.concat([
    sample_group(N_POS, diabetic=True),
    sample_group(N_NEG, diabetic=False),
], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

print("=" * 55)
print("  HEALTHCARE DATASET — DIABETES PREDICTION")
print("=" * 55)
print(f"\nShape      : {df.shape}")
print(f"Positive % : {df['diabetes'].mean():.1%}  ({df['diabetes'].sum()} diabetic patients)")
print(f"Negative % : {1 - df['diabetes'].mean():.1%}  ({(df['diabetes']==0).sum()} non-diabetic patients)")
print("\nSample rows:")
print(df.head(5).to_string(index=False))
print("\nDescriptive statistics:")
print(df.describe().round(2).to_string())

# ─────────────────────────────────────────────
# 2. PREPROCESSING
# ─────────────────────────────────────────────

FEATURES = [
    "age", "bmi", "glucose_level", "blood_pressure",
    "hba1c", "insulin_resistance", "family_history", "physical_activity"
]
TARGET = "diabetes"

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"\nTrain size : {len(X_train)} patients")
print(f"Test size  : {len(X_test)} patients")

# ─────────────────────────────────────────────
# 3. TRAIN & EVALUATE MODELS
# ─────────────────────────────────────────────

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree":       DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
}

results = {}

print("\n" + "=" * 55)
print("  MODEL TRAINING & EVALUATION")
print("=" * 55)

for name, model in models.items():
    if name == "Logistic Regression":
        model.fit(X_train_scaled, y_train)
        y_pred  = model.predict(X_test_scaled)
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
        cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring="accuracy")
    else:
        model.fit(X_train, y_train)
        y_pred  = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]
        cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring="accuracy")

    results[name] = {
        "model":     model,
        "y_pred":    y_pred,
        "y_proba":   y_proba,
        "accuracy":  accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall":    recall_score(y_test, y_pred, zero_division=0),
        "f1":        f1_score(y_test, y_pred, zero_division=0),
        "auc":       roc_auc_score(y_test, y_proba),
        "cv_mean":   cv_scores.mean(),
        "cv_std":    cv_scores.std(),
        "cm":        confusion_matrix(y_test, y_pred),
    }

    r = results[name]
    print(f"\n── {name} ──")
    print(f"  Accuracy  : {r['accuracy']:.4f}")
    print(f"  Precision : {r['precision']:.4f}")
    print(f"  Recall    : {r['recall']:.4f}")
    print(f"  F1 Score  : {r['f1']:.4f}")
    print(f"  AUC-ROC   : {r['auc']:.4f}")
    print(f"  CV Acc    : {r['cv_mean']:.4f} ± {r['cv_std']:.4f}")
    print(f"\n  Classification Report:\n")
    print(classification_report(y_test, y_pred,
                                target_names=["No Diabetes", "Diabetes"],
                                zero_division=0))

# Summary table
summary = pd.DataFrame({
    name: {
        "Accuracy":  f"{r['accuracy']:.4f}",
        "Precision": f"{r['precision']:.4f}",
        "Recall":    f"{r['recall']:.4f}",
        "F1 Score":  f"{r['f1']:.4f}",
        "AUC-ROC":   f"{r['auc']:.4f}",
        "5-Fold CV": f"{r['cv_mean']:.4f} ± {r['cv_std']:.4f}",
    }
    for name, r in results.items()
}).T

print("\n" + "=" * 55)
print("  COMPARISON SUMMARY TABLE")
print("=" * 55)
print(summary.to_string())

# ─────────────────────────────────────────────
# 4. VISUALIZATIONS
# ─────────────────────────────────────────────

COLORS = {
    "Logistic Regression": "#888780",
    "Decision Tree":       "#0F6E56",
    "Random Forest":       "#185FA5",
}

fig = plt.figure(figsize=(18, 22))
fig.suptitle("Healthcare ML Pipeline — Diabetes Prediction", fontsize=16, fontweight="bold", y=0.98)
gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.55, wspace=0.35)

# 4a. Confusion Matrices
for i, (name, r) in enumerate(results.items()):
    ax = fig.add_subplot(gs[0, i])
    sns.heatmap(r["cm"], annot=True, fmt="d", ax=ax,
                cmap="Blues", cbar=False,
                xticklabels=["Pred No", "Pred Yes"],
                yticklabels=["Actual No", "Actual Yes"])
    ax.set_title(f"Confusion Matrix\n{name}", fontsize=10, fontweight="bold")
    ax.set_xlabel("Predicted", fontsize=9)
    ax.set_ylabel("Actual", fontsize=9)

# 4b. ROC Curves
ax_roc = fig.add_subplot(gs[1, :2])
ax_roc.plot([0,1],[0,1], "k--", lw=1, label="Random baseline (AUC=0.50)")
for name, r in results.items():
    fpr, tpr, _ = roc_curve(y_test, r["y_proba"])
    ax_roc.plot(fpr, tpr, lw=2.5, color=COLORS[name],
                label=f"{name} (AUC={r['auc']:.3f})")
ax_roc.set_xlabel("False Positive Rate", fontsize=10)
ax_roc.set_ylabel("True Positive Rate", fontsize=10)
ax_roc.set_title("ROC Curves — All Models", fontsize=11, fontweight="bold")
ax_roc.legend(fontsize=9)
ax_roc.grid(alpha=0.3)

# 4c. Grouped Bar Chart
ax_bar = fig.add_subplot(gs[1, 2])
metrics     = ["accuracy", "precision", "recall", "f1", "auc"]
met_labels  = ["Accuracy", "Precision", "Recall", "F1", "AUC"]
x = np.arange(len(metrics))
w = 0.25
for i, (name, r) in enumerate(results.items()):
    vals = [r[m] for m in metrics]
    ax_bar.bar(x + i*w, vals, w, label=name, color=COLORS[name], alpha=0.85)
ax_bar.set_xticks(x + w)
ax_bar.set_xticklabels(met_labels, fontsize=8)
ax_bar.set_ylim(0.5, 1.05)
ax_bar.set_title("Metric Comparison", fontsize=11, fontweight="bold")
ax_bar.legend(fontsize=7)
ax_bar.grid(axis="y", alpha=0.3)

# 4d. Feature Importance — Random Forest
ax_fi = fig.add_subplot(gs[2, :2])
rf = results["Random Forest"]["model"]
importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)
bars = ax_fi.barh(importances.index, importances.values, color="#185FA5", alpha=0.8)
ax_fi.set_xlabel("Importance Score", fontsize=10)
ax_fi.set_title("Feature Importance — Random Forest", fontsize=11, fontweight="bold")
ax_fi.grid(axis="x", alpha=0.3)
for bar, val in zip(bars, importances.values):
    ax_fi.text(val + 0.001, bar.get_y() + bar.get_height()/2,
               f"{val:.3f}", va="center", fontsize=8)

# 4e. LR Coefficients
ax_coef = fig.add_subplot(gs[2, 2])
lr      = results["Logistic Regression"]["model"]
coefs   = pd.Series(lr.coef_[0], index=FEATURES).sort_values()
c_colors= ["#E24B4A" if c < 0 else "#185FA5" for c in coefs.values]
ax_coef.barh(coefs.index, coefs.values, color=c_colors, alpha=0.8)
ax_coef.axvline(0, color="black", lw=0.8)
ax_coef.set_xlabel("Coefficient", fontsize=10)
ax_coef.set_title("LR Coefficients\n(scaled features)", fontsize=10, fontweight="bold")
ax_coef.grid(axis="x", alpha=0.3)

# 4f. Decision Tree
ax_dt = fig.add_subplot(gs[3, :])
dt = results["Decision Tree"]["model"]
plot_tree(dt, feature_names=FEATURES,
          class_names=["No Diabetes", "Diabetes"],
          filled=True, rounded=True, max_depth=3, fontsize=7, ax=ax_dt)
ax_dt.set_title("Decision Tree Structure (max_depth=3 shown)", fontsize=11, fontweight="bold")

plt.savefig("/mnt/user-data/outputs/healthcare_ml_results.png", dpi=150, bbox_inches="tight")
print("\nPlot saved → healthcare_ml_results.png")
plt.show()

# ─────────────────────────────────────────────
# 5. SAVE DATASET
# ─────────────────────────────────────────────
df.to_csv("/mnt/user-data/outputs/healthcare_dataset.csv", index=False)
print("Dataset saved → healthcare_dataset.csv")

print("\n" + "=" * 55)
print("  PIPELINE COMPLETE")
print("=" * 55)